# Connecting with Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#Change working directory
%cd '/content/drive/MyDrive/Colab Notebooks/deep_acelerometry'

/content/drive/MyDrive/Colab Notebooks/deep_acelerometry


# Setup

In [3]:
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.utils import Sequence
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, ReLU, Add, MaxPooling2D, GlobalAveragePooling2D, Dense, Dropout, Flatten, Concatenate
from tensorflow.keras.regularizers import l2
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt

In [4]:
# Check that we are using GPU
tf.config.experimental.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

# Dataset

### 1. Load target labels

In [5]:
target = pd.read_csv('target.csv')

In [6]:
target

,SEQN,DXXNK_TSCORE,CLASSES,TARGET_BINARY
0,73557.0,NaN,NaN,NaN
1,73558.0,-0.358333,normal,0.0
2,73559.0,NaN,NaN,NaN
3,73561.0,-1.133333,osteopenia,1.0
4,73562.0,0.316667,normal,0.0
...,...,...,...,...
3703,83721.0,-0.266667,normal,0.0
3704,83723.0,1.191667,normal,0.0
3705,83724.0,NaN,NaN,NaN
3706,83726.0,0.841667,normal,0.0


### 2. Load relevant features for classification: gender, age and body mass.

In [7]:
X_train = pd.read_csv('X_train.csv')
X_test = pd.read_csv('X_test.csv')

In [8]:
X_train.shape, X_test.shape

((2003, 9), (501, 9))

In [9]:
X_train.iloc[:,:4]

,SEQN,RIAGENDR,RIDAGEYR,BMXWT
0,76988.0,1.0,67.0,68.3
1,82083.0,2.0,57.0,67.4
2,75435.0,2.0,63.0,60.3
3,76741.0,2.0,68.0,70.1
4,78834.0,2.0,56.0,57.1
...,...,...,...,...
1998,74345.0,1.0,56.0,79.2
1999,82126.0,1.0,56.0,70.5
2000,78475.0,2.0,77.0,71.7
2001,74862.0,2.0,48.0,66.1


In [10]:
# Combine them into a single DataFrame
features_df = pd.concat([X_train.iloc[:,:4], X_test.iloc[:,:4]]).reset_index(drop=True)

In [11]:
features_df

,SEQN,RIAGENDR,RIDAGEYR,BMXWT
0,76988.0,1.0,67.0,68.3
1,82083.0,2.0,57.0,67.4
2,75435.0,2.0,63.0,60.3
3,76741.0,2.0,68.0,70.1
4,78834.0,2.0,56.0,57.1
...,...,...,...,...
2499,79610.0,2.0,66.0,74.3
2500,77717.0,1.0,48.0,131.7
2501,75946.0,2.0,71.0,92.4
2502,75081.0,2.0,66.0,64.7


### 3. Physical activity data (tensors with GAF images)

Choose sampling frequency, which determines image size

In [12]:
# directory = '/content/drive/MyDrive/Colab Notebooks/deep_acelerometry/tensors_288' # 1 sample every 5 minutes
directory = '/content/drive/MyDrive/Colab Notebooks/deep_acelerometry/tensors_144' # 1 sample every 10 minutes
image_size = int(directory.split('_')[2])

# Read tensor filenames
all_files = [f for f in os.listdir(directory) if f.endswith('.npy')]

# Original 80/20 train-test split, already encoded in filenames
train_files = [f for f in all_files if f.startswith('train_')]
test_files = [f for f in all_files if f.startswith('test_')]

print("Total tensor files:", len(all_files))
print("Training tensor files:", len(train_files))
print("Test tensor files:", len(test_files))


Total tensor files: 2504
Training tensor files: 2003
Test tensor files: 501


In [13]:
image_size

144

# Data generator

### Step 1: Define the Generator Class
You'll subclass tf.keras.utils.Sequence to ensure your generator handles batch processing and shuffling properly.

In [14]:
class DataGenerator(Sequence):
    def __init__(self, directory, file_list, target_df, features_df, batch_size=32, shuffle=True, **kwargs):
        super().__init__(**kwargs)

        self.directory = directory
        self.file_list = file_list
        self.target_df = target_df
        self.features_df = features_df
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.file_list))

        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __len__(self):
        # Use ceil so the final incomplete batch is not dropped
        return int(np.ceil(len(self.file_list) / self.batch_size))

    def __getitem__(self, index):
        indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        batch_files = [self.file_list[k] for k in indexes]
        return self.__load_data(batch_files)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __load_data(self, batch_files):
        """Loads data for one batch."""
        x = []
        y = []
        features = []

        for file_name in batch_files:
            file_path = os.path.join(self.directory, file_name)
            data = np.load(file_path)
            x.append(data)

            subject_id = float(file_name.split('_')[1])

            # Regression target: femoral neck T-score
            target_value = self.target_df[
                self.target_df['SEQN'] == subject_id
            ]['DXXNK_TSCORE'].iloc[0]
            y.append(target_value)

            # Additional features: sex, age, body mass
            feature_row = self.features_df[
                self.features_df['SEQN'] == subject_id
            ][['RIAGENDR', 'RIDAGEYR', 'BMXWT']]

            if not feature_row.empty:
                features.append(feature_row.iloc[0].values)
            else:
                features.append(np.array([np.nan, np.nan, np.nan]))

        # Newer Keras expects multi-input data as a tuple, not a list.
        # Cast everything to float32 to avoid dtype errors.
        return (
            (np.array(x, dtype=np.float32), np.array(features, dtype=np.float32)),
            np.array(y, dtype=np.float32)
        )

### Step 2: Instantiate the Data Generator
Now you can create instances of DataGenerator for both training and validation.

In [15]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
batch_size = 32 # You can adjust this based on your GPU capacity and training needs

def extract_seqn_from_filename(file_name):
    """
    Extract participant SEQN from filenames such as:
    train_12345_144.npy or test_12345_144.npy
    """
    return float(file_name.split('_')[1])

# For regression, we use TARGET_BINARY only to preserve the normal vs
# osteopenia/osteoporosis distribution in the internal validation split.
train_strata = [
    target.loc[target['SEQN'] == extract_seqn_from_filename(f), 'TARGET_BINARY'].iloc[0]
    for f in train_files
]

# Split only the original training files.
# test_files remain untouched until final evaluation.
train_inner_files, val_files = train_test_split(
    train_files,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=train_strata
)

print("Internal training files:", len(train_inner_files))
print("Internal validation files:", len(val_files))
print("Final untouched test files:", len(test_files))

train_generator = DataGenerator(
    directory=directory,
    file_list=train_inner_files,
    target_df=target,
    features_df=features_df,
    batch_size=batch_size,
    shuffle=True
)

val_generator = DataGenerator(
    directory=directory,
    file_list=val_files,
    target_df=target,
    features_df=features_df,
    batch_size=batch_size,
    shuffle=False
)

test_generator = DataGenerator(
    directory=directory,
    file_list=test_files,
    target_df=target,
    features_df=features_df,
    batch_size=batch_size,
    shuffle=False
)

Internal training files: 1602
Internal validation files: 401
Final untouched test files: 501


# Auxiliary function

In [16]:
from tensorflow.keras import backend as K

# Define the R^2 metric
def r2_keras(y_true, y_pred):
    """
    Keras R² metric used during training.
    Final manuscript R² will be computed after training using sklearn on the test set.
    """
    y_true = K.cast(y_true, dtype='float32')
    y_pred = K.cast(y_pred, dtype='float32')

    ss_res = K.sum(K.square(y_true - y_pred))
    ss_tot = K.sum(K.square(y_true - K.mean(y_true)))

    return 1 - ss_res / (ss_tot + K.epsilon())

In [17]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def get_true_regression_values(generator):
    """
    Extract true continuous T-score values from a Keras Sequence generator.
    """
    y_true = []

    for i in range(len(generator)):
        _, y_batch = generator[i]
        y_true.extend(y_batch)

    return np.array(y_true, dtype=np.float32)


def evaluate_regression_model(model, test_generator, model_name):
    """
    Evaluate a regression model on the untouched test set.
    Outcome = femoral neck T-score.
    """
    y_true = get_true_regression_values(test_generator)
    y_pred = model.predict(test_generator).ravel().astype(np.float32)

    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    results = {
        "Model": model_name,
        "R2": r2,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae
    }

    predictions = pd.DataFrame({
        "y_true": y_true,
        "y_pred": y_pred,
        "residual": y_true - y_pred
    })

    return results, predictions

# Compile and train

**Modelo 1)** 5 veces Conv2D + BatchNormalization + MaxPooling

In [ ]:
# Define the input layer for the images
image_input = Input(shape=(image_size, image_size, 7))
# Define the input layer for the additional features
features_input = Input(shape=(3,))

# CNN layers - deeper and more complex
nof_filters = 64
x = Conv2D(nof_filters, (3, 3), activation='relu', padding='same')(image_input)
x = BatchNormalization()(x)
x = MaxPooling2D((2, 2))(x)
x = Conv2D(nof_filters * 2, (3, 3), activation='relu', padding='same')(x)
x = BatchNormalization()(x)
x = MaxPooling2D((2, 2))(x)
x = Conv2D(nof_filters * 4, (3, 3), activation='relu', padding='same')(x)
x = BatchNormalization()(x)
x = MaxPooling2D((2, 2))(x)
x = Conv2D(nof_filters * 8, (3, 3), activation='relu', padding='same')(x)
x = BatchNormalization()(x)
x = MaxPooling2D((2, 2))(x)
x = Conv2D(nof_filters * 16, (3, 3), activation='relu', padding='same')(x)
x = BatchNormalization()(x)
x = MaxPooling2D((2, 2))(x)
x = Dropout(0.1)(x)
x = Flatten()(x)

# Concatenate CNN output and additional features
x = Concatenate()([x, features_input])

# Dense layers
nof_neurons = 64
x = Dense(nof_neurons, activation='relu')(x)
x = Dropout(0.3)(x)
x = BatchNormalization()(x)
output = Dense(1, activation='linear')(x)  # Output layer for regression

# Create and compile the model
model = Model(inputs=[image_input, features_input], outputs=output)
#model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mean_absolute_error']) # Metrics for regression
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mean_squared_error',  r2_keras])


# Add callbacks for early stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Train the model
history = model.fit(train_generator, validation_data=test_generator, epochs=20, callbacks=[early_stopping])


Epoch 1/20
62/62 [==============================] - 26s 361ms/step - loss: 2.1759 - mean_squared_error: 2.1759 - r2_keras: -0.6573 - val_loss: 1.8028 - val_mean_squared_error: 1.8028 - val_r2_keras: -0.1887
Epoch 2/20
62/62 [==============================] - 21s 331ms/step - loss: 1.7637 - mean_squared_error: 1.7637 - r2_keras: -0.3233 - val_loss: 2.5337 - val_mean_squared_error: 2.5337 - val_r2_keras: -0.7374
Epoch 3/20
62/62 [==============================] - 21s 342ms/step - loss: 1.6621 - mean_squared_error: 1.6621 - r2_keras: -0.2143 - val_loss: 1.6455 - val_mean_squared_error: 1.6455 - val_r2_keras: -0.0509
Epoch 4/20
62/62 [==============================] - 21s 331ms/step - loss: 1.5477 - mean_squared_error: 1.5477 - r2_keras: -0.1499 - val_loss: 1.7699 - val_mean_squared_error: 1.7699 - val_r2_keras: -0.1767
Epoch 5/20
62/62 [==============================] - 22s 359ms/step - loss: 1.4512 - mean_squared_error: 1.4512 - r2_keras: -0.0534 - val_loss: 1.6540 - val_mean_squared_err

**Modelo 2)** ResNet

In [18]:
def residual_block(x, filters, kernel_size=3, stride=1, downsample=False):
    shortcut = x
    if downsample:
        shortcut = Conv2D(filters, 1, strides=stride, padding='same')(x)
        shortcut = BatchNormalization()(shortcut)

    x = Conv2D(filters, kernel_size, padding='same', strides=stride)(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = Conv2D(filters, kernel_size, padding='same')(x)
    x = BatchNormalization()(x)

    x = Add()([x, shortcut])
    x = ReLU()(x)
    return x

def tfm_resnet(input_shape, additional_features_shape):
    image_input = Input(shape=input_shape)
    features_input = Input(shape=additional_features_shape)

    # Initial convolution
    x = Conv2D(64, (7, 7), strides=2, padding='same')(image_input)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = MaxPooling2D((3, 3), strides=2, padding='same')(x)

    # ResNet blocks
    x = residual_block(x, 64)
    x = residual_block(x, 64)
    x = residual_block(x, 128, stride=2, downsample=True)
    x = residual_block(x, 128)
    x = residual_block(x, 256, stride=2, downsample=True)
    x = residual_block(x, 256)

    # Global Pooling and output layer for image features
    x = GlobalAveragePooling2D()(x)

    # Concatenate CNN output and additional features
    x = Concatenate()([x, features_input])

    # Dense layers and final regression layer
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    x = BatchNormalization()(x)
    output = Dense(1, activation='linear')(x)  # Change for regression

    # Create model
    model = Model(inputs=[image_input, features_input], outputs=output)
    return model

# Set up and compile the ResNet model for regression
resnet_model = tfm_resnet(
    input_shape=(image_size, image_size, 7),
    additional_features_shape=(3,)
)

resnet_model.compile(
    optimizer='adam',
    loss='mean_squared_error',
    metrics=['mean_squared_error', r2_keras]
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# Train using the internal validation set, not the final test set
resnet_history = resnet_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=[early_stopping]
)

# Final evaluation on the untouched test set
resnet_results, resnet_predictions = evaluate_regression_model(
    resnet_model,
    test_generator,
    "ResNet"
)

resnet_results


Epoch 1/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 1624s 32s/step - loss: 1.9929 - mean_squared_error: 1.9929 - r2_keras: -61.3862 - val_loss: 17.0624 - val_mean_squared_error: 17.0624 - val_r2_keras: -420.2002
Epoch 2/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 13s 263ms/step - loss: 1.5807 - mean_squared_error: 1.5807 - r2_keras: -55.2686 - val_loss: 1.6109 - val_mean_squared_error: 1.6109 - val_r2_keras: -51.8024
Epoch 3/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 13s 260ms/step - loss: 1.4917 - mean_squared_error: 1.4917 - r2_keras: -50.9203 - val_loss: 113.9603 - val_mean_squared_error: 113.9603 - val_r2_keras: -2651.6736
Epoch 4/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 13s 257ms/step - loss: 1.3109 - mean_squared_error: 1.3109 - r2_keras: -48.2318 - val_loss: 1535.8206 - val_mean_squared_error: 1535.8206 - val_r2_keras: -36300.8477
Epoch 5/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 13s 256ms/step - loss: 1.2310 - mean_squared_error: 1.2310 - r2_keras: -45.6727 - val_loss: 1.4391 - val_mean_squared_error: 1.4391 - val_r2_keras: -47.4047
Epoch 6

{'Model': 'ResNet',
 'R2': 0.11808687448501587,
 'MSE': 1.389732003211975,
 'RMSE': np.float64(1.1788689508219203),
 'MAE': 0.9575948119163513}

**Modelo 3)** VGG

In [19]:
# Define the input layer for the images
image_input = Input(shape=(image_size, image_size, 7), name='image_input')
# Define the input layer for the additional features
features_input = Input(shape=(3,), name='features_input')

# Block 1
x = Conv2D(64, (3, 3), activation='relu', padding='same', name='block1_conv1')(image_input)
x = Conv2D(64, (3, 3), activation='relu', padding='same', name='block1_conv2')(x)
x = MaxPooling2D((2, 2), strides=(2, 2), name='block1_pool')(x)

# Block 2
x = Conv2D(128, (3, 3), activation='relu', padding='same', name='block2_conv1')(x)
x = Conv2D(128, (3, 3), activation='relu', padding='same', name='block2_conv2')(x)
x = MaxPooling2D((2, 2), strides=(2, 2), name='block2_pool')(x)

# Block 3
x = Conv2D(256, (3, 3), activation='relu', padding='same', name='block3_conv1')(x)
x = Conv2D(256, (3, 3), activation='relu', padding='same', name='block3_conv2')(x)
x = Conv2D(256, (3, 3), activation='relu', padding='same', name='block3_conv3')(x)
x = MaxPooling2D((2, 2), strides=(2, 2), name='block3_pool')(x)

# Block 4
x = Conv2D(512, (3, 3), activation='relu', padding='same', name='block4_conv1')(x)
x = Conv2D(512, (3, 3), activation='relu', padding='same', name='block4_conv2')(x)
x = Conv2D(512, (3, 3), activation='relu', padding='same', name='block4_conv3')(x)
x = MaxPooling2D((2, 2), strides=(2, 2), name='block4_pool')(x)

# Flatten and concatenate additional features
x = Flatten(name='flatten')(x)
x = Concatenate()([x, features_input])

# Dense layers for regression
x = Dense(4096, activation='relu', name='fc1')(x)
x = Dropout(0.5)(x)  # Incorporating dropout for regularization
x = Dense(4096, activation='relu', name='fc2')(x)
x = Dropout(0.5)(x)  # Incorporating dropout for regularization
output = Dense(1, activation='linear', name='predictions')(x)  # Output layer for regression

# Create the VGG model
vgg_model = Model(inputs=[image_input, features_input], outputs=output)

vgg_model.compile(
    optimizer='adam',
    loss='mean_squared_error',
    metrics=['mean_squared_error', r2_keras]
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

vgg_model.summary()

# Train using the internal validation set, not the final test set
vgg_history = vgg_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=[early_stopping]
)

# Final evaluation on the untouched test set
vgg_results, vgg_predictions = evaluate_regression_model(
    vgg_model,
    test_generator,
    "VGG"
)

vgg_results

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 144, 144,  │          0 │ -                 │
│ (InputLayer)        │ 7)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1        │ (None, 144, 144,  │      4,096 │ image_input[0][0] │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2        │ (None, 144, 144,  │     36,928 │ block1_conv1[0][… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_pool         │ (None, 72, 72,    │          0 │ block1_conv2[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_conv1        │ (None, 72, 72,    │     73,856 │ block1_pool[0][0] │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_conv2        │ (None, 72, 72,    │    147,584 │ block2_conv1[0][… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_pool         │ (None, 36, 36,    │          0 │ block2_conv2[0][… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_conv1        │ (None, 36, 36,    │    295,168 │ block2_pool[0][0] │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_conv2        │ (None, 36, 36,    │    590,080 │ block3_conv1[0][… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_conv3        │ (None, 36, 36,    │    590,080 │ block3_conv2[0][… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_pool         │ (None, 18, 18,    │          0 │ block3_conv3[0][… │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block4_conv1        │ (None, 18, 18,    │  1,180,160 │ block3_pool[0][0] │
│ (Conv2D)            │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block4_conv2        │ (None, 18, 18,    │  2,359,808 │ block4_conv1[0][… │
│ (Conv2D)            │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block4_conv3        │ (None, 18, 18,    │  2,359,808 │ block4_conv2[0][… │
│ (Conv2D)            │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block4_pool         │ (None, 9, 9, 512) │          0 │ block4_conv3[0][… │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 41472)     │          0 │ block4_pool[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ features_input      │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                 

 Total params: 194,308,673 (741.23 MB)

 Trainable params: 194,308,673 (741.23 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 70s 804ms/step - loss: 83.2369 - mean_squared_error: 83.2369 - r2_keras: -1828.9419 - val_loss: 1.5062 - val_mean_squared_error: 1.5062 - val_r2_keras: -43.4216
Epoch 2/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 17s 329ms/step - loss: 1.6480 - mean_squared_error: 1.6480 - r2_keras: -54.5872 - val_loss: 1.1660 - val_mean_squared_error: 1.1660 - val_r2_keras: -38.8709
Epoch 3/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 17s 337ms/step - loss: 1.3828 - mean_squared_error: 1.3828 - r2_keras: -47.2554 - val_loss: 1.1395 - val_mean_squared_error: 1.1395 - val_r2_keras: -36.3771
Epoch 4/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 15s 297ms/step - loss: 1.2245 - mean_squared_error: 1.2245 - r2_keras: -151.4866 - val_loss: 1.1458 - val_mean_squared_error: 1.1458 - val_r2_keras: -35.5351
Epoch 5/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 16s 307ms/step - loss: 1.1830 - mean_squared_error: 1.1830 - r2_keras: -41.1330 - val_loss: 1.1416 - val_mean_squared_error: 1.1416 - val_r2_keras: -37.2556
Epoch 6/20
51/51 ━━━

{'Model': 'VGG',
 'R2': 0.3049231767654419,
 'MSE': 1.0953125953674316,
 'RMSE': np.float64(1.0465718300085436),
 'MAE': 0.7891218066215515}

In [20]:
regression_results = pd.DataFrame([
    resnet_results,
    vgg_results
])

regression_results

,Model,R2,MSE,RMSE,MAE
0,ResNet,0.118087,1.389732,1.178869,0.957595
1,VGG,0.304923,1.095313,1.046572,0.789122


In [21]:
regression_results.to_csv("cnn_regression_test_metrics.csv", index=False)

resnet_predictions.to_csv("resnet_regression_test_predictions.csv", index=False)
vgg_predictions.to_csv("vgg_regression_test_predictions.csv", index=False)

resnet_model.save("resnet_regression_final.keras")
vgg_model.save("vgg_regression_final.keras")